# 30 metrics (g5)

How long it takes. What you should see: every join and leave between the two newest EirGrid editions confirmed against the PDFs by ref and by name (the name check needs no ref pattern); then pick 10 and check them by hand in the two PDFs. Then each batch's distribution burn-down at the same ages, transmission waiting time, capacity by county and year first signed, projects on hold, and the exhibit.

In [ ]:
import subprocess

from iegrid import metrics, store
from iegrid.checks import transitions_table

snap = store.latest_snapshot()
sha = subprocess.run(["git", "-C", str(store.ROOT), "rev-parse", "--short", "HEAD"],
                     capture_output=True, text=True, check=True).stdout.strip()
con = store.connect_read_only(snap)
print(f"snapshot {snap.name}  git {sha}")

## done-when: transitions between the two newest editions, confirmed by the PDFs, 10 for a hand check

In [ ]:
rows = transitions_table(con)
for r in rows:
    print(f"{r[0]:15} {r[1]:26} {r[2]:7} {r[3]:7} {r[4]:40} {r[5]:7.1f} MW  by ref {r[6]}  by name {r[7]}")
unconfirmed = [r for r in rows if not (r[6] and r[7])]
assert rows, "no transitions"
assert not unconfirmed, f"transitions the PDFs don't confirm: {unconfirmed}"
print(f"\n{len(rows)} transitions, every one confirmed by ref and by name")
print("hand check, in the two PDFs:")
for r in rows[:: max(1, len(rows) // 10)][:10]:
    print(f"  {r[2]:7} {r[3]:7} {r[4]}")

## batch results dates (the earliest results publication stored)

In [ ]:
for b, d in metrics.batch_dates(con).items():
    print(f'{b:8} {d.label()}')

## distribution burn-down by batch, at the same ages

In [ ]:
ages = (12, 24, 36, 48, 60, 72)
print(f"{'batch':8} {'results':12} {'age':>4}  " + "  ".join(f"{a // 12}y".rjust(5) for a in ages) + "   now")
for b in metrics.burndown(con):
    cells = [b.share_at(a) for a in ages]
    print(f"{b.batch:8} {b.result.label():12} {b.age_months:4}  " +
          "  ".join(("-" if c is None else f"{c:.0%}").rjust(5) for c in cells) +
          f"  {b.share:5.0%}  ({b.energised_n} of {b.energised_n + b.contracted_n} projects)")

## transmission: how long projects still contracted have waited (from their earliest signed month)

In [ ]:
for w in metrics.tso_waiting(con):
    print(f"{w.route:26} {w.projects:3} projects {float(w.mw):7.0f} MW  median {w.median_months:5.1f} months  longest {w.longest_months}")

## capacity by county and year first signed (newest g4 build; year 0 = distribution, no signed date)

In [ ]:
from collections import defaultdict

grid = defaultdict(dict)
for county, year, mw in metrics.region_by_year(con):
    grid[county][year] = float(mw)
years = sorted({y for v in grid.values() for y in v})
print(f"{'county':10}" + "".join(f"{('dso' if y == 0 else y)!s:>7}" for y in years))
for county in sorted(grid, key=lambda c: -sum(grid[c].values())):
    print(f"{county:10}" + "".join(f"{grid[county].get(y, 0):7.0f}" for y in years))

## distribution projects on hold, per edition

In [ ]:
for sid, as_of, n, mw in metrics.on_hold(con):
    print(f"{sid:32} {as_of}  {n:3} projects  {float(mw):7.1f} MW")

## exhibit: how long it takes

In [ ]:
from iegrid import exhibits

fig = exhibits.batch_burndown(con)
exhibits.save(fig, "g5_batch_burndown")